In [0]:
#Função Info()

# Info dos Dados
# Tipo da Coluna
# Quantidade de linhas
# Quantidade de nulos
# Quantidade de Valores únicos
import pyspark.sql.functions as F
def info(x):

    # Número total de linhas
    n_rows = x.count()

    summary = []
    for c in x.columns:
        dtype = dict(x.dtypes)[c]
        n_nulls = x.filter(F.col(c).isNull()).count()
        n_uniques = x.select(c).distinct().count()
        summary.append((c, dtype, n_rows, n_nulls, n_uniques))

    # Crie o DataFrame de resumo
    summary_df = spark.createDataFrame(
        summary,
        ["coluna", "tipo", "qtd_linhas", "qtd_nulos", "qtd_valores_unicos"]
    )

    summary_df.show()

# The Transforming Logic

In [0]:
query = """

SELECT sha2(CAST(crmcust.customer_id AS STRING), 256) AS customer_key,
        crmcust.customer_id,
       erpcust.customer_number as customer_number_erp,
       crmcust.first_name,
       CASE 
        WHEN crmcust.gender = "Unknown" THEN erpcust.gender
        ELSE COALESCE(crmcust.gender, "Unknown")
       END as gender,
       crmcust.marital_status,
       erplo.country
     
FROM workspace.silver.crm_customers AS crmcust

LEFT JOIN workspace.silver.erp_customers AS erpcust
    ON crmcust.customer_number = erpcust.customer_number

LEFT JOIN workspace.silver.erp_customer_location AS erplo
    ON crmcust.customer_number = erplo.customer_number



"""

df = spark.sql(query)
display(df.limit(5))
info(df)

#df.groupBy(F.col("gender_final")).count().show()
#df.groupBy(F.col("gender")).count().show()
#df.groupBy(F.col("gender_erp")).count().show()


# Write into Golde Layer

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.gold.dim_customers")

In [0]:
%skip
query_1 = """
SELECT *
FROM workspace.silver.crm_customers
"""

df_erp = spark.sql(query_1)
df_erp.show(n=5)

query_2 = """
SELECT *
FROM workspace.silver.erp_customer_location
"""

df_erp_lo = spark.sql(query_2)
df_erp_lo.show(n=5)